In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
ratings = pd.read_csv("data/ratings.csv", nrows=1000000)
movies = pd.read_csv("data/movies.csv")

df = ratings.merge(movies, on="movieId", how="left")

df["year"] = df["title"].str.extract(r'(\d{4})').astype(float)
df.loc[df["year"] > 2025, "year"] = np.nan

In [4]:
user_stats = df.groupby("userId")["rating"].agg(
    avg_rating_user="mean",
    num_ratings_user="count"
).reset_index()

movie_stats = df.groupby("movieId")["rating"].agg(
    avg_rating_movie="mean",
    num_ratings_movie="count"
).reset_index()

df2 = df.merge(user_stats, on="userId", how="left")
df2 = df2.merge(movie_stats, on="movieId", how="left")

In [5]:
df2["genre_Drama"] = df2["genres"].str.contains("Drama", na=False).astype(int)
df2["genre_Comedy"] = df2["genres"].str.contains("Comedy", na=False).astype(int)
df2["genre_Action"] = df2["genres"].str.contains("Action", na=False).astype(int)

In [6]:
df2["num_ratings_user"] = np.log1p(df2["num_ratings_user"])
df2["num_ratings_movie"] = np.log1p(df2["num_ratings_movie"])

In [7]:
features = [
    "avg_rating_movie",
    "avg_rating_user",
    "num_ratings_movie",
    "num_ratings_user",
    "genre_Drama",
    "genre_Comedy",
    "genre_Action",
    "year"
]

X = df2[features].fillna(0)
y = df2["rating"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2:", round(r2, 4))

MAE: 0.66
RMSE: 0.8576
R2: 0.3345


In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [0.6600],
    "RMSE": [0.8576],
    "R2": [0.3345],
    "Member": ["Tuân"]
})

results.to_csv("linear_regression_results_tuan.csv", index=False)